# RG3 상태별 D 보완 저장/로드 및 평가 라벨 의존 검사


In [ ]:
from pathlib import Path
import nbformat,joblib,json,hashlib
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
ROOT=Path.cwd();WORK=ROOT/'rg3/internal';tables=WORK/'reports/tables'
s=''.join(c.source for c in nbformat.read(WORK/'notebooks/209_rg3_shortshot_state_conditioned_D.ipynb',4).cells if c.cell_type=='code')
ns={'__name__':'__main__'};exec(s[:s.index('records=[];audit=[]')],ns)
# Fresh heldout-label perturbation on the block containing4 short shots.
g=ns['g'];y=ns['y'];use=ns['use'];lab=ns['lab']
q=np.flatnonzero(g=='rg3_R3');ref=np.flatnonzero((g!='rg3_R3')&use)
a=ns['predict'](ref,q).copy()
ns['y']=y.copy();ns['y'][q]=1-ns['y'][q];ns['memo']={}
b=ns['predict'](ref,q).copy()
assert np.max(abs(a-b))<1e-12
flip_diff=float(np.max(abs(a-b)));ns['y']=y;ns['memo']={}
# Recompute all fullfit calibration scores from block-excluded training, not unknown predictions.
ref=np.flatnonzero(use);inner=np.full(len(ref),np.nan)
for ib in pd.unique(g[ref]):
 mask=g[ref]==ib;inner[mask]=ns['predict'](ref[~mask],ref[mask])
cut=float(np.quantile(inner[y[ref]==0],.995,method='higher'))
normal_ids=ref[y[ref]==0];select=ns['pool_selection'](ref)
refs=np.vstack([ns['joint'][normal_ids],ns['pool_joint'][select]])
ctx=np.vstack([ns['context'][normal_ids],ns['pool_context'][select]])
times=np.r_[lab.TimeStamp.to_numpy()[normal_ids],ns['pool_times'][select]]
ci=SimpleImputer(strategy='median').fit(ctx);cx=ci.transform(ctx);sc=RobustScaler().fit(cx)
ri=SimpleImputer(strategy='median').fit(refs);rx=ri.transform(refs)
peer={'ci':ci,'sc':sc,'ri':ri,'raw':rx,'timestamps':times,'nn':NearestNeighbors(n_neighbors=min(101,len(rx))).fit(sc.transform(cx)),'floor':np.maximum(rx.std(axis=0)*.05,np.array([.1,.1,.1,.1,.1,.01,.01,.01,.01,.01,.01]))}
def conditioned_saved(peer,joint,context,times):
 query=peer['ri'].transform(joint);near=peer['nn'].kneighbors(peer['sc'].transform(peer['ci'].transform(context)),return_distance=False)
 out=[]
 for row,ids,t in zip(query,near,times):
  ids=ids[peer['timestamps'][ids]!=np.datetime64(t)][:100]
  center=np.median(peer['raw'][ids],axis=0)
  spread=np.maximum(1.4826*np.median(abs(peer['raw'][ids]-center),axis=0),peer['floor'])
  out.append(np.clip((row-center)/spread,-12,12))
 return np.asarray(out)
trainX=conditioned_saved(peer,ns['joint'][ref],ns['context'][ref],lab.TimeStamp.to_numpy()[ref])
assert np.allclose(trainX,ns['conditioned'](ref,ns['joint'][ref],ns['context'][ref],lab.TimeStamp.to_numpy()[ref]),atol=1e-12)
model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(ns['features'](trainX),y[ref])
# Persist original-temperature applicability statistics; this isn't a quality classifier.
temp=ns['context'][normal_ids];ti=SimpleImputer(strategy='median').fit(temp);temp=ti.transform(temp)
med=np.median(temp,axis=0);scale=np.maximum(np.quantile(temp,.75,axis=0)-np.quantile(temp,.25,axis=0),.1)
nn=NearestNeighbors(n_neighbors=2).fit((temp-med)/scale)
support_cut=float(np.quantile(nn.kneighbors((temp-med)/scale)[0][:,1],.99,method='higher'))
old=joblib.load(WORK/'models/reduced_shortshot15/shortshot_best.joblib')
bundle={'format':'RG3_SHORTSHOT_ROUTED_D_V1','original15':old,'conditioned_peer':peer,'D_model':model,'D_cut':cut,'support':{'imputer':ti,'median':med,'scale':scale,'nn':nn,'cut':support_cut},'features':['max_joint','second_joint','mean_joint','joint_count'],'reference_cutoff':str(ns['cutoff']),'reference_unknown_count':int(select.sum()),'classifiers':16}
def routed_predict(bundle,f,joint,context,timestamps):
 # The legacy feature transformer is loaded as definitions only, without training.
 package=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4)
 helper={'__name__':'__main__'};exec(package.cells[1].source,helper)
 _,scores=helper['predict_short'](bundle['original15'],f)
 cuts=bundle['original15']['cuts']
 abc=(scores[:,0]>=cuts['A'])|(scores[:,1]>=cuts['B'])|((scores[:,2]>=cuts['C'])&(f.Barrel_Temperature_5__past10_std.to_numpy()>=cuts['barrel5']))
 oldD=(scores[:,3]>=cuts['D'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])
 p=bundle['support'];dist=p['nn'].kneighbors((p['imputer'].transform(context)-p['median'])/p['scale'],n_neighbors=1)[0][:,0];supported=dist<=p['cut']
 z=conditioned_saved(bundle['conditioned_peer'],joint,context,timestamps)
 score=bundle['D_model'].predict_proba(ns['features'](z))[:,1]
 newD=(score>=bundle['D_cut'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])
 return abc|np.where(supported,oldD,newD),score,supported
u=ns['u']
f=u[['TimeStamp','Barrel_Temperature_5__past10_std','Hopper_Temperature__dev10']].copy()
for key,arr in [('std',u[[v+'__current_std10' for v in old['sensor_names']]].to_numpy(float)),('physical',ns['phyU']),('joint',ns['jointU']),('context',u[['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']].to_numpy(float)),('shock',np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Cushion_Position__past10_std]))]:
 for j,col in enumerate(old['columns'][key]):f[col]=arr[:,j]
pred,score,sup=routed_predict(bundle,f,ns['jointU'],ns['ctxU'],u.TimeStamp.to_numpy())
expected=pd.read_csv(tables/'rg3_210_fallback_0.995_unlabeled.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[u.TimeStamp]
assert np.array_equal(pred,expected.shortshot_prediction.to_numpy(bool))
folder=WORK/'models/shortshot_routed_D_candidate';folder.mkdir(parents=True,exist_ok=True)
path=folder/'shortshot_routed_D.joblib';joblib.dump(bundle,path,compress=3)
loaded=joblib.load(path);pred2,score2,sup2=routed_predict(loaded,f,ns['jointU'],ns['ctxU'],u.TimeStamp.to_numpy())
assert np.array_equal(pred,pred2) and np.allclose(score,score2,atol=1e-12)
summary={'classifiers':16,'old_D_within_supported_state':True,'new_D_outside_supported_state':True,'query_label_flip_maxdiff':flip_diff,'unlabeled':len(u),'candidates':int(pred.sum()),'candidate_percent':float(100*pred.mean()),'all_unlabeled_predictions_match_210':True,'save_load_match':True,'past_reference_unknown_rows':int(select.sum()),'reference_cutoff':str(ns['cutoff']),'reference_selection_train_labels_only':True,'unlabeled_used_as_class_targets':False,'OOF_TP':6,'OOF_FP':11,'OOF_FN':2,'OOF_TN':555,'OOF_F1':.48,'model_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'deployment_replaced':False,'pushed':False}
(folder/'verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(summary)

